# EYES-DEFY-ANEMIA -- EfficientNet-B0 (forniceal_palpebral) -- 5-Fold Cross-Validation

Dedicated follow-up pipeline for the single best batch-1 performer (val F1=0.933). Fresh random head init per fold (no warm-start), hyperparameters locked from batch-1 trial #9 (no Optuna), gradient clipping, `ReduceLROnPlateau` + `EarlyStopping`, batch size 32, 150-epoch ceiling, unchanged augmentation baseline (flip + rotate only). The 33-patient test set is permanently excluded from all 5 folds. Full rationale in `classification/.project_memory/02_current_status.md`.

Each fold runs as its own cell (`--fold N`), not one long call, specifically so a mid-run Kaggle session interruption only loses the fold in progress -- the same incremental-safety pattern as `classification-final-fixed.ipynb` (batch 1) and `classification-batch2.ipynb`.

## Setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia
!git pull  # belt-and-suspenders after a fresh clone -- see memory notes for why this is a safe no-op normally

In [ ]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

In [ ]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations

## Data

In [ ]:
import shutil
from pathlib import Path

SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

In [ ]:
import sys

sys.path.insert(0, "classification/scripts/efficientnet_b0_forniceal_5fold_cv")
from cv_dataset import load_cv_pool, build_folds, N_FOLDS

pool = load_cv_pool()
print(f"CV pool: {len(pool)} patients (expect 178 -- 33-patient test set excluded)")
assert len(pool) == 178, f"expected 178, got {len(pool)}"

folds = build_folds(pool, n_folds=N_FOLDS)
print(f"Built {len(folds)} folds:")
for i, (train_df, val_df) in enumerate(folds, start=1):
    print(f"  Fold {i}: train={len(train_df)} val={len(val_df)}")

## sync_outputs() -- consolidate + zip after every fold

Same design as batch 1/2: copies `classification/scripts/efficientnet_b0_forniceal_5fold_cv/outputs/{checkpoints,logs,plots}/` into a top-level `/kaggle/working/outputs/` and zips it to `/kaggle/working/efficientnet_b0_cv_results.zip`. Called after every one of the 6 training cells below (5 folds + 1 aggregation step), not just at the end.

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate this pipeline's own outputs/{checkpoints,logs,plots}/
    into a top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/efficientnet_b0_cv_results.zip. Called after every
    fold (and the final aggregation step) so a mid-run interruption --
    a real risk over 5 folds x up to 150 epochs, by far the longest run
    in this project -- still leaves a complete, downloadable snapshot
    of whatever finished."""
    src_dir = Path("classification/scripts/efficientnet_b0_forniceal_5fold_cv/outputs")
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        sub_src = src_dir / sub
        if sub_src.exists():
            shutil.copytree(sub_src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive(
        "/kaggle/working/efficientnet_b0_cv_results", "zip", root_dir=str(results_dir)
    )
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

## Training -- 5 folds, then aggregate

Each cell is a separate `!python` call to `run_cv_training.py`, one fold at a time (`--fold N`), then a final `--aggregate` pass that reads all 5 folds' saved history and computes the cross-fold summary -- no re-training. A failed cell does not halt "Run All" (same IPython behavior as batch 1/2) -- check each cell's own output, or the saved `.../outputs/logs/efficientnet_b0_forniceal_palpebral_cv_fold*_history.json` files, after this finishes.

In [ ]:
# Fold 1 / 5
!python classification/scripts/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 1
sync_outputs()

In [ ]:
# Fold 2 / 5
!python classification/scripts/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 2
sync_outputs()

In [ ]:
# Fold 3 / 5
!python classification/scripts/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 3
sync_outputs()

In [ ]:
# Fold 4 / 5
!python classification/scripts/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 4
sync_outputs()

In [ ]:
# Fold 5 / 5
!python classification/scripts/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 5
sync_outputs()

In [ ]:
# Aggregate -- reads all 5 folds' saved history, computes the cross-fold summary. No training.
!python classification/scripts/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --aggregate
sync_outputs()

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever folds completed, plus `efficientnet_b0_forniceal_palpebral_cv_cv_summary.json` if the aggregate step ran) and zipped to `/kaggle/working/efficientnet_b0_cv_results.zip`. Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All -- download the zip directly from there, or browse the folder for individual files.

In [ ]:
print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/efficientnet_b0_cv_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")